# Solar Filament Segmentation — Foundation (Phase 0 & 1) on Google Colab

Runs the foundation checks on a cloud GPU:
1. Clone repo (dataset is inside the repo, ~720 MB)
2. Install dependencies + build Detectron2
3. Freeze exact versions -> `requirements-lock.txt`
4. Run the test suite (RLE round-trip, PQ metric, frozen split)
5. EDA
6. **End-to-end dry run**: 5 training iterations -> val PQ -> test inference -> validated `submission_dryrun.csv` -> `experiments/log.csv`

**Before you start:** `Runtime -> Change runtime type -> T4 GPU` (or better).
If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) holding a read-only personal access token.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Clone the repository

In [ ]:
import os
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if not os.path.exists("/content/solar"):
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1
!ls filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images | wc -l   # expect 707
!ls filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images | wc -l    # expect 180

## 2. Install dependencies (Detectron2 build takes ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'

## 3. Freeze the exact environment (for reproducibility / final `requirements.txt`)

In [ ]:
import subprocess, sys, torch, detectron2
keep = ("torch", "torchvision", "numpy", "detectron2", "pycocotools", "opencv", "albumentations",
        "torchmetrics", "scikit-learn", "matplotlib", "seaborn", "pandas", "tqdm", "scikit-image",
        "scipy", "pillow", "pytest")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
lines = [l for l in freeze.splitlines() if l.lower().split("==")[0].split(" @ ")[0].startswith(keep)]
header = f"# Colab {torch.cuda.get_device_name(0)} | Python {sys.version.split()[0]} | CUDA {torch.version.cuda}"
with open("requirements-lock.txt", "w") as f:
    f.write(chr(10).join([header] + lines) + chr(10))
print(open("requirements-lock.txt").read())

## 4. Test suite (expect `14 passed`)

In [ ]:
!python -m pytest -q -p no:warnings tests

## 5. EDA (also computes the inter-annotator PQ ceiling)

In [ ]:
!python -W ignore src/eda.py --no-images
from IPython.display import Image, display
display(Image("output/eda/filament_stats.png"))

## 6. End-to-end dry run (Phase 0 exit criterion)
5 iterations on 8 train images, PQ on 4 val images, inference on 3 test images.
PQ will be ~0 — this only proves the plumbing works. A 5-iteration model may predict
no masks at all; that is fine (the CSV is then header-only and validation is skipped).

In [ ]:
!python run_pipeline.py --dry-run

In [ ]:
!cat experiments/log.csv
!head -c 600 submissions/submission_dryrun.csv; echo
!cat output/val_results_dryrun.json

## 7. Send results back
Download these and paste/commit them into the repo (or paste the printed output into Claude):
- `requirements-lock.txt`
- `experiments/log.csv` (the dry-run row)

In [ ]:
from google.colab import files
files.download("requirements-lock.txt")
files.download("experiments/log.csv")

## 8. (On Kaggle, not Colab) Metric alignment with the official self-evaluation notebook
Phase 1 exit criterion: our `src/evaluate.py` must match the organizers' notebook within ±0.001 PQ.
1. Open https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook -> **Copy & Edit**.
2. Note which ground truth and prediction file it reads, then run it unchanged and record its PQ / Dice.
3. Copy its PQ-computation cell(s) and send them to Claude, together with the numbers it printed,
   so `src/evaluate.py` can be checked for: pooled vs per-image PQ, `IoU > 0.5` vs `>=`, and how
   overlapping predictions are matched.